# User-Cohort Dynamics

Describe changes in account-creation cohorts and user characteristics across Stack Exchange sites. The current monthly export is grouped by account creation month and therefore does **not** measure monthly active users.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt

def root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        q=p/'stack_exchange_analysis'
        if (p/'database').exists(): return p
        if (q/'database').exists(): return q
    raise FileNotFoundError
PROJECT_ROOT=root(); DATA_DIR=PROJECT_ROOT/'database'; ANALYSIS_DIR=PROJECT_ROOT/'analysis'; ANALYSIS_DIR.mkdir(exist_ok=True); sys.path.insert(0,str(PROJECT_ROOT/'src'))
from analysis_utils import read_csv_flexible, save_figure

users=read_csv_flexible(DATA_DIR/'user-metrics-by-site-by-month-all-time.csv'); month_col=next((c for c in users if c.lower() in {'creationmonth','month','date'}),None); site_col=next((c for c in users if c.lower()=='site'),None)
if month_col is None or site_col is None: raise ValueError(f'Expected Site and CreationMonth-like columns, got {list(users.columns)}')
users[month_col]=pd.to_datetime(users[month_col],errors='coerce')
for c in users.columns:
    if c not in {month_col,site_col}: users[c]=pd.to_numeric(users[c],errors='coerce')
print(users[[site_col,month_col]].dropna().agg({month_col:['min','max']}))

## Stack Overflow registration cohorts

In [ ]:
so=users[users[site_col].astype(str).str.contains('StackOverflow',case=False,na=False)].sort_values(month_col).copy(); display(so.tail())
if not so.empty and 'UsersCount' in so:
    fig,ax=plt.subplots(figsize=(12,4)); ax.plot(so[month_col],so['UsersCount']); ax.set(title='Stack Overflow account-creation cohorts',xlabel='Creation month',ylabel='Users'); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/'users_stackoverflow_cohorts.png'); plt.show()

## Cohort characteristics

In [ ]:
metrics=[c for c in ['AvgReputation','AvgProfileViews','AvgUpVotesCast','AvgDownVotesCast','AvgDaysSinceLastAccess','AvgAccountAgeDays'] if c in so]
if metrics: display(so[[month_col]+metrics].tail(24))
for metric in metrics[:4]:
    fig,ax=plt.subplots(figsize=(12,3.5)); ax.plot(so[month_col],so[metric]); ax.set(title=f'Stack Overflow cohort metric: {metric}',xlabel='Creation month',ylabel=metric); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/f'users_{metric.lower()}.png'); plt.show()

## Cross-site registration trajectories

In [ ]:
if 'UsersCount' in users:
    p=users.pivot_table(index=month_col,columns=site_col,values='UsersCount',aggfunc='sum').sort_index()
    def norm(s):
        pos=s[s>0].dropna(); b=pos.iloc[:12].mean() if len(pos)>=6 else np.nan; return 100*s/b if pd.notna(b) and b else s*np.nan
    idx=p.apply(norm); display(idx.dropna(how='all').iloc[-1].sort_values().rename('latest_registration_index').to_frame())

## Static site-level metrics

In [ ]:
static=read_csv_flexible(DATA_DIR/'user-static-metrics-by-several-site-all-time.csv'); display(static.sort_values('UsersCount',ascending=False).head(25) if 'UsersCount' in static else static.head(25))

## Takeaways
These data support investigation of entry and cohort composition. Active askers, active answerers, first-time askers, and retention require new activity-based SQL exports before participation mechanisms can be evaluated.